# UN Speeches — Topic Classification
Run on GPU runtime: **Runtime → Change runtime type → T4 GPU**

Upload these files before running:
- `speeches_paragraphs.csv`  (from `data/`)
- `topic_definitions.csv`    (from `raw_data/`)
- `clean_countries.py`       (from `streamlit/`)

In [ ]:
# ── Cell 1: Upload files ───────────────────────────────────────────────────────
from google.colab import files
print('Upload speeches_paragraphs.csv, topic_definitions.csv, and clean_countries.py')
uploaded = files.upload()

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q sentence-transformers umap-learn scikit-learn pandas

In [ ]:
# ── Cell 3: Verify GPU ─────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 4: Run classification ─────────────────────────────────────────────────
import ast
import sys
import numpy as np
import pandas as pd
import umap
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, '/content')
from clean_countries import clean_country, to_drop

DATA_PATH               = Path('/content/speeches_paragraphs.csv')
TOPICS_PATH             = Path('/content/topic_definitions.csv')
TOPIC_LABELS_OUT        = Path('/content/topic_labels.csv')
UMAP_OUT                = Path('/content/speeches_umap.csv')
MENTIONED_COUNTRIES_OUT = Path('/content/mentioned_countries_agg.csv')

BATCH_SIZE = 512   # larger batch = faster on GPU
MODEL_NAME = 'all-mpnet-base-v2'

# ── Load data ──────────────────────────────────────────────────────────────────
print('Loading data…')
df        = pd.read_csv(DATA_PATH)
topics_df = pd.read_csv(TOPICS_PATH)
topic_names = topics_df['Name'].str.strip().tolist()
topic_texts = topics_df['Text'].str.strip().tolist()
print(f'  {len(df):,} paragraphs')
print(f'  {len(topic_names)} topics:', topic_names)

# ── Encode ─────────────────────────────────────────────────────────────────────
print(f'\nLoading model ({MODEL_NAME})…')
model = SentenceTransformer(MODEL_NAME)

print('Encoding topic anchors…')
anchor_embeddings = model.encode(topic_texts, show_progress_bar=False)

print(f'Encoding {len(df):,} paragraphs (batch_size={BATCH_SIZE})…')
paragraphs = df['speeches'].fillna('').tolist()
paragraph_embeddings = model.encode(
    paragraphs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# ── Assign topics ──────────────────────────────────────────────────────────────
print('Computing similarities…')
similarities = cosine_similarity(paragraph_embeddings, anchor_embeddings)
best_idx     = similarities.argmax(axis=1)
best_score   = similarities.max(axis=1)

df['topic']            = [topic_names[i] for i in best_idx]
df['topic_similarity'] = best_score.round(4)
df.to_csv(DATA_PATH, index=False)
print('\n✅ Updated speeches_paragraphs.csv')

print('\nTopic distribution:')
print(df['topic'].value_counts().to_string())

low_confidence = (best_score < 0.25).sum()
print(f'\nLow-confidence rows (similarity < 0.25): {low_confidence:,} ({low_confidence / len(df) * 100:.1f}%)')

In [ ]:
# ── Cell 5: Regenerate topic_labels.csv ───────────────────────────────────────
rows = []
for _, row in topics_df.iterrows():
    name     = row['Name']
    count    = int((df['topic'] == name).sum())
    keywords = [kw.strip() for kw in str(row['Short_keywords']).split(',')]
    rows.append({
        'topic_id':    int(row['Index']),
        'topic_name':  name,
        'count':       count,
        'top_5_words': str(keywords[:5]),
    })
pd.DataFrame(rows).to_csv(TOPIC_LABELS_OUT, index=False)
print(f'✅ Saved topic_labels.csv ({len(rows)} topics)')

In [ ]:
# ── Cell 6: Regenerate UMAP embeddings ────────────────────────────────────────
print('Running UMAP (CPU, takes a few minutes)…')
reducer     = umap.UMAP(n_components=2, random_state=42)
umap_coords = reducer.fit_transform(paragraph_embeddings)
umap_df     = pd.DataFrame(umap_coords, columns=['umap_1', 'umap_2'])
combined    = pd.concat(
    [df[['iso', 'year', 'country', 'continent', 'topic']].reset_index(drop=True), umap_df],
    axis=1
)
speeches_umap = (
    combined
    .groupby(['iso', 'year', 'country', 'continent', 'topic'], dropna=False)
    .agg(umap_1=('umap_1', 'mean'), umap_2=('umap_2', 'mean'), count=('umap_1', 'count'))
    .reset_index()
)
speeches_umap[['umap_1', 'umap_2']] = speeches_umap[['umap_1', 'umap_2']].round(4)
speeches_umap.to_csv(UMAP_OUT, index=False)
print(f'✅ Saved speeches_umap.csv ({len(speeches_umap):,} rows)')

In [ ]:
# ── Cell 7: Regenerate mentioned_countries_agg.csv ────────────────────────────
mc = df[df['topic'] != 'bla_bla'][['year', 'topic', 'country', 'countries_recoded']].copy()
mc['countries_recoded'] = mc['countries_recoded'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) and x != '[]' else []
)
exploded = mc.explode('countries_recoded').rename(columns={'countries_recoded': 'country_mentioned'})
exploded = exploded[exploded['country_mentioned'].notna()]
exploded['country_mentioned'] = exploded['country_mentioned'].apply(clean_country)
exploded = exploded[~exploded['country_mentioned'].isin(to_drop)]
result = exploded.groupby(['year', 'topic', 'country_mentioned']).size().reset_index(name='country_count')
result.to_csv(MENTIONED_COUNTRIES_OUT, index=False)
print(f'✅ Saved mentioned_countries_agg.csv ({len(result):,} rows)')

In [ ]:
# ── Cell 8: Download outputs ───────────────────────────────────────────────────
from google.colab import files
for path in [DATA_PATH, TOPIC_LABELS_OUT, UMAP_OUT, MENTIONED_COUNTRIES_OUT]:
    print(f'Downloading {path.name}…')
    files.download(str(path))
print('Done! Copy the downloaded files back to their original locations:')
print('  speeches_paragraphs.csv   → data/')
print('  topic_labels.csv          → raw_data/')
print('  speeches_umap.csv         → streamlit/')
print('  mentioned_countries_agg.csv → streamlit/')